# Silver Transformation — TfL Stop Points

Transform raw TfL StopPoint snapshots into structured Silver datasets.

This notebook:

1. Reads Bronze StopPoint snapshots.
2. Parses the nested JSON using an explicit schema.
3. Creates a normalized stop-point dataset.
4. Creates a stop-point-to-line relationship dataset.
5. Applies data-quality rules.
6. Deduplicates records.
7. Uses Delta `MERGE` for idempotent writes.

**Source:** `workspace.urbanpulse_bronze.tfl_stop_points`

**Targets:**

- `workspace.urbanpulse_silver.tfl_stop_points`
- `workspace.urbanpulse_silver.tfl_stop_point_lines`

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import transformation and quality components

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.tfl_stop_points import (
    parse_tfl_stop_points,
    transform_stop_points,
    transform_stop_point_lines,
)

from urbanpulse.quality.tfl_stop_points import (
    valid_stop_points,
    invalid_stop_points,
)

from urbanpulse.utils.delta import (
    merge_insert_only,
)

## 3. Define source and target tables

In [0]:
BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "tfl_stop_points"
)

SILVER_STOP_POINTS = (
    "workspace."
    "urbanpulse_silver."
    "tfl_stop_points"
)

SILVER_STOP_POINT_LINES = (
    "workspace."
    "urbanpulse_silver."
    "tfl_stop_point_lines"
)

## 4. Read Bronze snapshots

In [0]:
bronze_df = spark.table(
    BRONZE_TABLE
)

bronze_count = bronze_df.count()

print(
    f"Bronze snapshots: {bronze_count}"
)

display(
    bronze_df.select(
        "request_id",
        "ingested_at",
        "http_status",
    )
)

## 5. Parse the TfL StopPoint payload

The raw JSON object contains a nested `stopPoints` array.

The explicit Spark schema converts that array into structured Spark data before individual stop points are exploded into rows.

In [0]:
parsed_df = parse_tfl_stop_points(
    bronze_df
)

parsed_count = parsed_df.count()

print(
    f"Parsed stop-point rows: "
    f"{parsed_count}"
)

In [0]:
display(
    parsed_df.select(
        "request_id",
        "stop_point.id",
        "stop_point.commonName",
        "stop_point.lat",
        "stop_point.lon",
        "stop_point.lines",
    )
)

## 6. Create the structured stop-point dataset

Extract core station attributes from the parsed source data.

In [0]:
stop_points_df = transform_stop_points(
    parsed_df
)

display(stop_points_df)

## 7. Validate stop-point records

Valid stop points require:

- request ID
- TfL stop-point ID
- station name
- valid latitude
- valid longitude
- snapshot timestamp

In [0]:
valid_df = valid_stop_points(
    stop_points_df
)

invalid_df = invalid_stop_points(
    stop_points_df
)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print(f"Valid rows: {valid_count}")
print(f"Invalid rows: {invalid_count}")

## 8. Enforce the stop-point data contract

Invalid core reference records cause the Silver transformation to fail rather than silently entering downstream datasets.

In [0]:
if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"Data quality failure: "
        f"{invalid_count} invalid stop points"
    )

print("Stop-point quality checks passed.")

## 9. Deduplicate stop points

Within each API snapshot, a stop point must be unique by:

`request_id + stop_point_id`

In [0]:
stop_points_deduped_df = (
    valid_df
    .dropDuplicates([
        "request_id",
        "stop_point_id",
    ])
    .withColumn(
        "processed_at",
        F.current_timestamp(),
    )
)

print(
    "Stop points after deduplication:",
    stop_points_deduped_df.count(),
)

## 10. Create stop-point-to-line relationships

TfL stores line information as an array inside each stop point.

Exploding this array creates a normalized relationship table that can later support station-line joins and Gold dimensional modelling.

In [0]:
stop_point_lines_df = (
    transform_stop_point_lines(
        parsed_df
    )
    .filter(
        F.col("stop_point_id").isNotNull()
    )
    .filter(
        F.col("line_id").isNotNull()
    )
    .dropDuplicates([
        "request_id",
        "stop_point_id",
        "line_id",
    ])
    .withColumn(
        "processed_at",
        F.current_timestamp(),
    )
)

print(
    "Stop-point-line relationships:",
    stop_point_lines_df.count(),
)

display(stop_point_lines_df)

## 11. Merge stop points into Silver

Delta `MERGE` makes the operation idempotent.

Reprocessing the same Bronze request does not create duplicate Silver records.

In [0]:
stop_points_result = merge_insert_only(
    spark=spark,
    source_df=stop_points_deduped_df,
    target_table=SILVER_STOP_POINTS,
    merge_condition="""
        target.request_id = source.request_id
        AND target.stop_point_id = source.stop_point_id
    """,
)

print(
    f"Stop points Silver table: "
    f"{stop_points_result}"
)

## 12. Merge stop-point-line relationships into Silver

In [0]:
line_result = merge_insert_only(
    spark=spark,
    source_df=stop_point_lines_df,
    target_table=SILVER_STOP_POINT_LINES,
    merge_condition="""
        target.request_id = source.request_id
        AND target.stop_point_id = source.stop_point_id
        AND target.line_id = source.line_id
    """,
)

print(
    f"Stop-point-line Silver table: "
    f"{line_result}"
)

## 13. Verify Silver stop points

In [0]:
%sql
SELECT
    stop_point_id,
    station_naptan,
    common_name,
    stop_type,
    latitude,
    longitude,
    modes,
    snapshot_at
FROM workspace.urbanpulse_silver.tfl_stop_points
ORDER BY common_name;

In [0]:
## 14. Verify stop-point-line relationships

In [0]:
%sql
SELECT
    stop_point_id,
    line_id,
    line_name,
    snapshot_at
FROM workspace.urbanpulse_silver.tfl_stop_point_lines
ORDER BY line_name, stop_point_id;

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT request_id) AS snapshots,
    COUNT(DISTINCT stop_point_id) AS stop_points
FROM workspace.urbanpulse_silver.tfl_stop_points;

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT stop_point_id) AS stop_points,
    COUNT(DISTINCT line_id) AS lines
FROM workspace.urbanpulse_silver.tfl_stop_point_lines;

In [0]:
%sql
SELECT
    request_id,
    stop_point_id,
    COUNT(*) AS records
FROM workspace.urbanpulse_silver.tfl_stop_points
GROUP BY
    request_id,
    stop_point_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    request_id,
    stop_point_id,
    line_id,
    COUNT(*) AS records
FROM workspace.urbanpulse_silver.tfl_stop_point_lines
GROUP BY
    request_id,
    stop_point_id,
    line_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.urbanpulse_silver.tfl_stop_points;

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.urbanpulse_silver.tfl_stop_point_lines;